In [ ]:
!pip install scikit-learn==1.4.2 -q

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
filepath = '/content/drive/MyDrive/Datasets/PhiUSIIL_Phishing_URL_Dataset.csv'

In [ ]:
df = pd.read_csv(filepath)

In [ ]:
print(df.head())

In [ ]:
df.tail(10)

In [ ]:
df.shape

In [ ]:
df.size

In [ ]:
df.info()

In [ ]:
for i in df:
  print(df.describe())

**Preprocessing Data**

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
print(df['Title'].value_counts().head(20))
print(f"\nTotal unique Title values: {df['Title'].nunique()}")

In [ ]:
Y = df['label']
X_df = df.drop(columns=['label', 'URLSimilarityIndex', 'URL', 'Domain', 'Title'])

# Now split into numeric and categorical AFTER cleaning
num_df         = X_df.select_dtypes(include=['number'])
categorical_df = X_df.select_dtypes(include=['object']) 

print("Categorical columns:", categorical_df.columns.tolist())
print("Numeric columns:", num_df.columns.tolist())
print("Label leaking into X?", 'label' in X_df.columns)  


In [ ]:
preprossesor = ColumnTransformer([
    ('one_hot_encoder', OneHotEncoder(handle_unknown='ignore'), categorical_df.columns),
    ('scaler', StandardScaler(), num_df.columns)
], remainder='passthrough')

In [ ]:
X = preprossesor.fit_transform(X_df)

In [ ]:
print(X)

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_temp,Y_train, Y_temp = train_test_split(X, Y, test_size=0.4, random_state=42)

In [ ]:
X_temp.shape

In [ ]:
X_train.shape

In [ ]:
Y_temp.shape

In [ ]:
Y_train.shape

**Validation Data**

In [ ]:
X_test,X_val,Y_test,Y_val = train_test_split(X_temp,Y_temp,test_size=0.5,random_state=42)

In [ ]:
X_test.shape

In [ ]:
X_val.shape

In [ ]:
Y_val.shape

**Training Model**

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
model  = XGBClassifier(objective='binary:logistic',n_estimators=10, max_depth =11,learning_rate=0.1, random_state=42)

In [ ]:
model.fit(X_train,Y_train)

**Perform Inference**

In [ ]:
result = model.score(X_test,Y_test)

In [ ]:
print('Accuracy{}'.format(result))

In [ ]:
y_predict = model.predict(X_test)
print('Label prediction:' , y_predict)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(Y_test,y_predict))

In [ ]:
import seaborn as sns

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test, y_predict)
sns.heatmap(cm, fmt = 'd', annot = True)

**validation Accuracy**

In [ ]:
Validation_result = model.score(X_val,Y_val)

In [ ]:
print('Validation Accuracy{}'.format(Validation_result))

In [ ]:
val_predict = model.predict(X_val)
print('Label prediction:' , val_predict)

In [ ]:
print(classification_report(Y_val,val_predict))

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_val, val_predict)
sns.heatmap(cm, fmt = 'd', annot = True)

In [ ]:
import joblib

# Get feature names from OneHotEncoder for categorical columns
one_hot_features = preprossesor.named_transformers_['one_hot_encoder'].get_feature_names_out(categorical_df.columns)

# Get numerical column names from the original num_df
numerical_features = num_df.columns

# Combine all feature names in the order ColumnTransformer produces them
all_transformed_features = list(one_hot_features) + list(numerical_features)
